# 🚀 CAVAT - Remote Dev Environment Setup

**Mục đích:** Thiết lập kết nối SSH từ VS Code (Local) tới Kaggle GPU thông qua TCP Tunnel.

**Dự án:** CAVAT - Hệ thống AI Trợ giảng Toán học (Qwen 2.5 + QLoRA + Custom RAG)

**Tunnel:** Sử dụng [bore](https://github.com/ekzhang/bore) — TCP tunnel mã nguồn mở, miễn phí, không cần đăng ký.

---

### ⚡ Hướng dẫn sử dụng:
1. Đảm bảo **GPU T4** đã bật (Settings → Accelerator → GPU T4 x2)
2. Bật **Internet** (Settings → Internet → On)
3. Chạy các cell **tuần tự** từ trên xuống dưới
4. Sau Cell 3, copy lệnh SSH hiển thị và dùng để kết nối từ VS Code

## 📋 Cell 1: Kiểm tra GPU & Môi trường

In [ ]:
# ============================================
# CELL 1: KIỂM TRA GPU & MÔI TRƯỜNG HỆ THỐNG
# ============================================

import torch
import subprocess
import sys
import os

print("="*60)
print("🔍 KIỂM TRA MÔI TRƯỜNG HỆ THỐNG CAVAT")
print("="*60)

# Kiểm tra GPU
print("\n📊 [GPU STATUS]")
!nvidia-smi

print(f"\n🔧 [PYTORCH INFO]")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   GPU Name:        {gpu_name}")
    print(f"   GPU Memory:      {gpu_mem:.1f} GB")
    print(f"   CUDA Version:    {torch.version.cuda}")
    print(f"\n   ✅ GPU sẵn sàng cho training Qwen 2.5 + QLoRA!")
else:
    print("\n   ❌ KHÔNG TÌM THẤY GPU!")
    print("   → Vào Settings → Accelerator → Chọn GPU T4 x2")
    print("   → Restart notebook sau khi thay đổi")

# Kiểm tra Python
print(f"\n🐍 [PYTHON INFO]")
print(f"   Python version:  {sys.version}")
print(f"   Working dir:     {os.getcwd()}")
print(f"   Disk space:")
!df -h /kaggle/working | tail -1

print("\n" + "="*60)
print("✅ Kiểm tra hoàn tất - Tiếp tục Cell 2")
print("="*60)

## 🔧 Cell 2: Cài đặt SSH Server + Bore Tunnel

In [ ]:
# ============================================
# CELL 2: CÀI ĐẶT SSH SERVER + BORE TUNNEL
# ============================================

import os

print("="*60)
print("🔧 CÀI ĐẶT SSH SERVER & BORE TUNNEL")
print("="*60)

# --- Bước 2.1: Cài đặt OpenSSH Server ---
print("\n📦 [1/4] Cài đặt OpenSSH Server...")
!apt-get update -qq && apt-get install -y -qq openssh-server > /dev/null 2>&1
print("   ✅ OpenSSH Server đã cài đặt")

# --- Bước 2.2: Cấu hình SSH ---
print("\n🔐 [2/4] Cấu hình SSH Server...")
!mkdir -p /var/run/sshd

# Đặt mật khẩu root
!echo 'root:cavat123' | chpasswd

# Cho phép root login qua SSH
!sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config

# Cho phép password authentication
!sed -i 's/#PasswordAuthentication yes/PasswordAuthentication yes/' /etc/ssh/sshd_config
!sed -i 's/PasswordAuthentication no/PasswordAuthentication yes/' /etc/ssh/sshd_config

print("   ✅ SSH đã cấu hình (Root login: YES, Password auth: YES)")

# --- Bước 2.3: Khởi chạy SSH ---
print("\n🚀 [3/4] Khởi chạy SSH Server...")
!service ssh start
print("   ✅ SSH Server đang chạy trên port 22")

# --- Bước 2.4: Tải bore tunnel ---
print("\n🌐 [4/4] Tải bore tunnel...")
BORE_VERSION = "v0.5.2"
if not os.path.exists('./bore'):
    !wget -q https://github.com/ekzhang/bore/releases/download/{BORE_VERSION}/bore-{BORE_VERSION}-x86_64-unknown-linux-musl.tar.gz -O bore.tar.gz
    !tar xzf bore.tar.gz
    !rm -f bore.tar.gz
    !chmod +x bore
    print("   ✅ bore đã tải và giải nén")
else:
    print("   ✅ bore đã tồn tại, bỏ qua tải")

# Kiểm tra
!./bore --version

print("\n" + "="*60)
print("✅ Cài đặt hoàn tất - Tiếp tục Cell 3")
print("="*60)

## 🌐 Cell 3: Khởi chạy Bore Tunnel & Hiển thị Endpoint

**bore** là TCP tunnel mã nguồn mở — **KHÔNG cần tài khoản, KHÔNG cần thẻ, hoàn toàn FREE.**

In [ ]:
# ============================================
# CELL 3: KHỞI CHẠY BORE TUNNEL
# ============================================

import subprocess
import time
import re
import json
import threading
import os

print("="*60)
print("🌐 KHỞI CHẠY BORE SSH TUNNEL")
print("="*60)

# Kill bore cũ nếu có
print("\n🔄 [1/2] Dọn dẹp tunnel cũ...")
!pkill -f 'bore local' 2>/dev/null || true
time.sleep(1)
print("   ✅ Đã dọn dẹp")

# Khởi chạy bore tunnel
print("\n🚇 [2/2] Mở SSH Tunnel...")
print("   ⏳ Đang kết nối tới bore.pub...")

# Chạy bore trong background và capture output
bore_process = subprocess.Popen(
    ['./bore', 'local', '22', '--to', 'bore.pub'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Đọc output để lấy port
tunnel_port = None
bore_log = []

def read_bore_output():
    global tunnel_port
    for line in bore_process.stdout:
        line = line.strip()
        bore_log.append(line)
        # bore output: "listening at bore.pub:XXXXX"
        match = re.search(r'bore\.pub:(\d+)', line)
        if match:
            tunnel_port = match.group(1)

reader_thread = threading.Thread(target=read_bore_output, daemon=True)
reader_thread.start()

# Chờ tunnel sẵn sàng (tối đa 15 giây)
for i in range(15):
    if tunnel_port:
        break
    time.sleep(1)

if tunnel_port:
    host = "bore.pub"
    port = tunnel_port
    
    print("\n" + "🟢"*30)
    print("\n" + "="*60)
    print("   🎉 BORE TUNNEL ĐÃ SẴN SÀNG!")
    print("="*60)
    print(f"")
    print(f"   🔗 Host:     {host}")
    print(f"   🔗 Port:     {port}")
    print(f"   🔑 Password: cavat123")
    print(f"")
    print(f"   ┌──────────────────────────────────────────────────┐")
    print(f"   │  📋 LỆNH KẾT NỐI (copy vào Terminal):          │")
    print(f"   │                                                  │")
    print(f"   │  ssh root@{host} -p {port:<27s}│")
    print(f"   │                                                  │")
    print(f"   └──────────────────────────────────────────────────┘")
    print(f"")
    print(f"   📝 VS Code SSH Config (~/.ssh/config):")
    print(f"   ┌──────────────────────────────────────────────────┐")
    print(f"   │  Host kaggle-cavat                               │")
    print(f"   │      HostName {host:<35s}│")
    print(f"   │      Port {port:<38s}│")
    print(f"   │      User root                                   │")
    print(f"   │      StrictHostKeyChecking no                    │")
    print(f"   │      UserKnownHostsFile /dev/null                │")
    print(f"   └──────────────────────────────────────────────────┘")
    print("\n" + "🟢"*30)
    
    # Lưu endpoint
    with open('/tmp/bore_endpoint.json', 'w') as f:
        json.dump({'host': host, 'port': port}, f)
    
    print("\n   ⚠️  QUAN TRỌNG: Giữ cell này ĐANG CHẠY — đừng dừng nó!")
    print("   (bore tunnel chạy foreground, dừng cell = mất kết nối)")
    
    # Giữ cell chạy để duy trì tunnel
    try:
        bore_process.wait()
    except KeyboardInterrupt:
        print("\n   🛑 Tunnel đã dừng bởi người dùng.")
else:
    print("\n   ❌ KHÔNG THỂ KHỞI TẠO TUNNEL!")
    print(f"")
    print(f"   📋 Bore log:")
    for log_line in bore_log:
        print(f"      {log_line}")
    print(f"")
    print(f"   🔍 Cách khắc phục:")
    print(f"   1. Kiểm tra Internet đã bật → Settings → Internet → On")
    print(f"   2. bore.pub có thể tạm thời quá tải → Chờ 1 phút rồi chạy lại")
    print(f"   3. Thử chạy thủ công: !./bore local 22 --to bore.pub")

## 🐕 Cell 4: Watchdog Heartbeat (Giám sát tự động)

In [ ]:
# ============================================
# CELL 4: WATCHDOG HEARTBEAT - GIÁM SÁT TỰ ĐỘNG
# ============================================

import threading
import time
import os
import subprocess
from datetime import datetime

print("="*60)
print("🐕 KHỞI CHẠY WATCHDOG HEARTBEAT")
print("="*60)

# Cấu hình
HEARTBEAT_INTERVAL = 60  # Kiểm tra mỗi 60 giây
MAX_SSH_RESTARTS = 5     # Tối đa 5 lần restart SSH

class WatchdogMonitor:
    def __init__(self):
        self.running = True
        self.ssh_restarts = 0
        self.check_count = 0
        self.last_bore_ok = True
        
    def check_ssh(self):
        """Kiểm tra SSH server còn chạy không"""
        try:
            result = os.popen("service ssh status 2>&1").read()
            return "running" in result.lower() or "active" in result.lower()
        except:
            return False
    
    def check_bore(self):
        """Kiểm tra bore tunnel còn hoạt động không"""
        try:
            result = subprocess.run(['pgrep', '-f', 'bore local'], capture_output=True)
            return result.returncode == 0
        except:
            return False
    
    def restart_ssh(self):
        """Khởi động lại SSH server"""
        os.system("service ssh start")
        self.ssh_restarts += 1
    
    def heartbeat(self):
        """Vòng lặp kiểm tra chính"""
        while self.running:
            self.check_count += 1
            timestamp = datetime.now().strftime("%H:%M:%S")
            
            ssh_ok = self.check_ssh()
            bore_ok = self.check_bore()
            
            # Status icons
            ssh_icon = "✅" if ssh_ok else "❌"
            bore_icon = "✅" if bore_ok else "❌"
            
            if ssh_ok and bore_ok:
                if self.check_count % 5 == 0:
                    print(f"[{timestamp}] #{self.check_count:04d} | SSH: {ssh_icon} | Bore: {bore_icon} | Uptime: {self.check_count} min")
            else:
                print(f"[{timestamp}] #{self.check_count:04d} | SSH: {ssh_icon} | Bore: {bore_icon} | ⚠️ CẢNH BÁO!")
                
                if not ssh_ok:
                    if self.ssh_restarts < MAX_SSH_RESTARTS:
                        print(f"   🔄 SSH DOWN → Đang restart (lần {self.ssh_restarts + 1}/{MAX_SSH_RESTARTS})...")
                        self.restart_ssh()
                    else:
                        print(f"   🛑 SSH đã restart {MAX_SSH_RESTARTS} lần, cần kiểm tra thủ công!")
                
                if not bore_ok:
                    if self.last_bore_ok:
                        print(f"   🚨 BORE TUNNEL ĐÃ MẤT KẾT NỐI!")
                        print(f"   → Chạy lại Cell 3 để tạo tunnel mới")
                        print(f"   → Sau đó cập nhật SSH config với port mới")
            
            self.last_bore_ok = bore_ok
            time.sleep(HEARTBEAT_INTERVAL)

# Khởi chạy watchdog
watchdog = WatchdogMonitor()
watchdog_thread = threading.Thread(target=watchdog.heartbeat, daemon=True)
watchdog_thread.start()

print(f"\n🐕 Watchdog đã khởi chạy thành công!")
print(f"   • Interval:      {HEARTBEAT_INTERVAL}s")
print(f"   • Max SSH retry: {MAX_SSH_RESTARTS}")
print(f"   • Status log:    Mỗi 5 phút")
print(f"   • Alert:         Ngay lập tức khi lỗi")
print(f"\n   ℹ️  Để dừng watchdog: watchdog.running = False")
print("\n" + "="*60)
print("✅ Watchdog đang giám sát - Tiếp tục Cell 5")
print("="*60)

## 📁 Cell 5: Clone Project & Thiết lập Git Auto-save

In [ ]:
# ============================================
# CELL 5: CLONE PROJECT & GIT AUTO-SAVE
# ============================================

import os
import time

print("="*60)
print("📁 THIẾT LẬP PROJECT & GIT AUTO-SAVE")
print("="*60)

GIT_REPO = "https://github.com/minhvuogdzz/MathAI-support-Project.git"
GIT_USER = "minhvuogdzz"
GIT_EMAIL = "minhvuogdzz@users.noreply.github.com"
PROJECT_DIR = "/kaggle/working/CAVAT_PROJECT"

# --- Bước 5.1: Cấu hình Git toàn cục ---
print("\n🔧 [1/3] Cấu hình Git...")
!git config --global user.name "{GIT_USER}"
!git config --global user.email "{GIT_EMAIL}"
!git config --global init.defaultBranch main
print(f"   ✅ Git user: {GIT_USER}")

# --- Bước 5.2: Clone hoặc Init project ---
print(f"\n📦 [2/3] Thiết lập project tại {PROJECT_DIR}...")

if os.path.exists(PROJECT_DIR):
    print(f"   ℹ️  Thư mục đã tồn tại, pull latest...")
    os.chdir(PROJECT_DIR)
    !git pull origin main 2>/dev/null || echo "   ℹ️  Không có thay đổi mới"
else:
    clone_result = os.system(f"git clone {GIT_REPO} {PROJECT_DIR} 2>/dev/null")
    if clone_result != 0:
        print("   ℹ️  Repo trống, khởi tạo local và kết nối remote...")
        os.makedirs(PROJECT_DIR, exist_ok=True)
        os.chdir(PROJECT_DIR)
        !git init
        !git remote add origin {GIT_REPO}
    else:
        os.chdir(PROJECT_DIR)

print(f"   ✅ Project directory: {PROJECT_DIR}")

# --- Bước 5.3: Hiển thị cấu trúc ---
print(f"\n📂 [3/3] Cấu trúc project hiện tại:")
!find . -maxdepth 2 -not -path './.git/*' -not -name '.git' | head -30

print("\n" + "="*60)
print("✅ Project đã sẵn sàng - Tiếp tục Cell 6")
print("="*60)

## 💾 Cell 6: Auto-save Functions (Gọi trước khi đóng session)

In [ ]:
# ============================================
# CELL 6: AUTO-SAVE FUNCTIONS
# ============================================

import os
import time
import threading
from datetime import datetime

PROJECT_DIR = "/kaggle/working/CAVAT_PROJECT"

def save_now(message=None):
    """
    💾 Lưu toàn bộ thay đổi lên GitHub ngay lập tức.
    Sử dụng: save_now() hoặc save_now("message")
    """
    os.chdir(PROJECT_DIR)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if message is None:
        message = f"Auto-save session {timestamp}"
    else:
        message = f"{message} [{timestamp}]"
    
    print(f"\n💾 Đang lưu: {message}")
    os.system("git add -A")
    
    status = os.popen("git status --porcelain").read().strip()
    if not status:
        staged = os.popen("git diff --cached --name-only").read().strip()
        if not staged:
            print("   ℹ️  Không có thay đổi mới để lưu.")
            return
    
    commit_result = os.system(f'git commit -m "{message}"')
    if commit_result != 0:
        print("   ℹ️  Không có thay đổi mới để commit.")
        return
    
    push_result = os.system("git push -u origin main")
    if push_result == 0:
        print(f"   ✅ Đã push thành công lên GitHub!")
    else:
        print(f"   ❌ Push thất bại!")
        print(f"      → git remote set-url origin https://<TOKEN>@github.com/minhvuogdzz/MathAI-support-Project.git")


def auto_save_periodic(interval_minutes=30):
    """⏰ Tự động lưu định kỳ. Sử dụng: auto_save_periodic(30)"""
    def _loop():
        count = 0
        while True:
            time.sleep(interval_minutes * 60)
            count += 1
            print(f"\n⏰ [Auto-save #{count}]")
            save_now(f"Periodic auto-save #{count}")
    
    threading.Thread(target=_loop, daemon=True).start()
    print(f"⏰ Auto-save đã bật: lưu mỗi {interval_minutes} phút")


print("="*60)
print("💾 AUTO-SAVE FUNCTIONS ĐÃ SẴN SÀNG")
print("="*60)
print(f"")
print(f"   save_now()              → Lưu ngay")
print(f"   save_now('message')     → Lưu với ghi chú")
print(f"   auto_save_periodic(30)  → Tự động lưu mỗi 30 phút")
print(f"")
print(f"   ⚠️  Luôn gọi save_now() trước khi đóng session!")
print("\n" + "="*60)
print("🎉 SETUP HOÀN TẤT! Kết nối VS Code ngay bây giờ.")
print("="*60)

---

## 📌 Quick Reference

| Hành động | Lệnh |
|-----------|-------|
| Kết nối SSH | `ssh root@bore.pub -p <PORT>` |
| Mật khẩu | `cavat123` |
| Lưu code | `save_now()` hoặc `save_now('message')` |
| Auto-save | `auto_save_periodic(30)` |
| Kiểm tra GPU | `nvidia-smi` |
| Dừng watchdog | `watchdog.running = False` |
| Restart tunnel | Dừng Cell 3 → Chạy lại Cell 3 |